# ⚡ Adaptive Flow & Beat Engine - Colab Stem Processor (GPU Edition)

Notebook này xử lý tách nhạc hàng loạt (Batch Processing) sử dụng GPU miễn phí của Google Colab (Tesla T4 / V100).

### 📁 Cấu trúc thư mục trên Google Drive:
- `AdaptiveMusic_Sync/01_Input_Audio/` : Nhạc do App đẩy lên cần tách
- `AdaptiveMusic_Sync/02_Processed_Audio/` : File gốc sau khi xử lý được chuyển về đây
- `AdaptiveMusic_Sync/03_Output_Stems/{video_id}/` : Lưu `calm_stem.mp3` & `energy_stem.mp3` (320kbps) + `metadata.json`

👉 **Hướng dẫn:** Chọn menu `Runtime` -> `Run all` (hoặc nhấn `Ctrl + F9`) và ngồi nhâm nhi cà phê!

In [ ]:
# 1. Kiểm tra GPU và Mount Google Drive
import os
import sys
import torch
from google.colab import drive

print("Đang kết nối Google Drive...")
drive.mount('/content/drive')

print("\n--- THÔNG TIN PHẦN CỨNG ---")
if torch.cuda.is_available():
    print(f"✅ GPU Kích hoạt: {torch.cuda.get_device_name(0)}")
    print(f"⚡ Bộ nhớ VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn T4 GPU")

In [ ]:
# 2. Cài đặt các thư viện AI & Âm thanh
!pip install -q demucs librosa soundfile pydub

In [ ]:
# 3. Khởi tạo cấu trúc thư mục đồng bộ
BASE_DIR = "/content/drive/MyDrive/AdaptiveMusic_Sync"
INPUT_DIR = os.path.join(BASE_DIR, "01_Input_Audio")
PROCESSED_DIR = os.path.join(BASE_DIR, "02_Processed_Audio")
OUTPUT_DIR = os.path.join(BASE_DIR, "03_Output_Stems")

for d in [INPUT_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Thư mục đồng bộ đã sẵn sàng tại: {BASE_DIR}")

In [ ]:
# 4. BATCH PROCESSOR: Tách nhạc HTDemucs + Lấy tên bài hát + Đo BPM + Nén MP3 320kbps
import glob
import json
import shutil
import subprocess
import urllib.request
import librosa
from pydub import AudioSegment

input_files = glob.glob(os.path.join(INPUT_DIR, "*.mp3")) + glob.glob(os.path.join(INPUT_DIR, "*.wav"))
total = len(input_files)

print(f"🎵 Tìm thấy {total} bài hát đang chờ trong 01_Input_Audio/\n")

if total == 0:
    print("Không có bài hát nào cần xử lý. App của bạn đã đồng bộ đầy đủ!")

for idx, file_path in enumerate(input_files, 1):
    filename = os.path.basename(file_path)
    video_id = os.path.splitext(filename)[0]
    print(f"\n[{idx}/{total}] 🚀 Bắt đầu xử lý: {filename} (ID: {video_id})")

    target_stem_dir = os.path.join(OUTPUT_DIR, video_id)
    os.makedirs(target_stem_dir, exist_ok=True)

    temp_demucs_dir = f"/content/temp_demucs_{video_id}"
    os.makedirs(temp_demucs_dir, exist_ok=True)

    try:
        # 1. Chạy HTDemucs trên GPU xuất WAV không nén
        cmd = [
            "python3", "-m", "demucs.separate",
            "-n", "htdemucs",
            "-d", "cuda" if torch.cuda.is_available() else "cpu",
            "-o", temp_demucs_dir,
            file_path
        ]
        subprocess.run(cmd, check=True)

        stems_folder = os.path.join(temp_demucs_dir, "htdemucs", os.path.splitext(filename)[0])
        drums_wav = os.path.join(stems_folder, "drums.wav")
        bass_wav = os.path.join(stems_folder, "bass.wav")
        other_wav = os.path.join(stems_folder, "other.wav")
        vocals_wav = os.path.join(stems_folder, "vocals.wav")

        # 2. Đo BPM & Phổ âm năng lượng bằng Librosa trên file gốc
        print("  📊 Đang phân tích BPM và phổ âm...")
        y, sr = librosa.load(file_path, sr=22050, duration=90) # mẫu 90 giây
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        bpm = float(tempo[0]) if hasattr(tempo, '__iter__') else float(tempo)
        rms_energy = float(librosa.feature.rms(y=y).mean())
        spectral_centroid = float(librosa.feature.spectral_centroid(y=y, sr=sr).mean())

        # 3. Xuất 4 stems độc lập chất lượng cao MP3 320kbps
        print("  🎛️ Đang xuất 4 stems độc lập (vocals, other, bass, drums) MP3 320kbps...")
        other_seg = AudioSegment.from_wav(other_wav)
        drums_seg = AudioSegment.from_wav(drums_wav)
        bass_seg = AudioSegment.from_wav(bass_wav)
        vocals_seg = AudioSegment.from_wav(vocals_wav)

        min_len = min(len(other_seg), len(drums_seg), len(bass_seg), len(vocals_seg))
        other_seg = other_seg[:min_len]
        drums_seg = drums_seg[:min_len]
        bass_seg = bass_seg[:min_len]
        vocals_seg = vocals_seg[:min_len]

        vocals_seg.export(os.path.join(target_stem_dir, "vocals.mp3"), format="mp3", bitrate="320k")
        other_seg.export(os.path.join(target_stem_dir, "other.mp3"), format="mp3", bitrate="320k")
        bass_seg.export(os.path.join(target_stem_dir, "bass.mp3"), format="mp3", bitrate="320k")
        drums_seg.export(os.path.join(target_stem_dir, "drums.mp3"), format="mp3", bitrate="320k")

        # 4. Lấy tiêu đề chính thức từ YouTube oEmbed và ghi metadata.json
        track_title = f"Track {video_id}"
        try:
            oembed_url = f"https://www.youtube.com/oembed?url=https://www.youtube.com/watch?v={video_id}&format=json"
            req = urllib.request.Request(oembed_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=4) as r:
                d = json.loads(r.read().decode())
                if d.get("title"):
                    track_title = d["title"]
        except Exception:
            pass

        meta = {
            "video_id": video_id,
            "title": track_title,
            "duration": int(min_len / 1000),
            "bpm": round(bpm, 1),
            "rms_energy": round(rms_energy, 4),
            "spectral_centroid": round(spectral_centroid, 2),
            "bitrate": "320kbps",
            "processed_by": "Google Colab HTDemucs GPU",
            "status": "READY"
        }
        with open(os.path.join(target_stem_dir, "metadata.json"), "w", encoding="utf-8") as mf:
            json.dump(meta, mf, indent=2, ensure_ascii=False)

        # 5. Di chuyển file gốc sang 02_Processed_Audio để tránh xử lý lại
        shutil.move(file_path, os.path.join(PROCESSED_DIR, filename))
        print(f"  ✅ Hoàn tất bài: {track_title} (BPM: {round(bpm, 1)}, Bitrate: 320kbps)")

    except Exception as e:
        print(f"  ❌ Lỗi xử lý {filename}: {e}")
    finally:
        shutil.rmtree(temp_demucs_dir, ignore_errors=True)

print("\n🎉 XONG TOÀN BỘ DANH SÁCH! Google Drive của bạn sẽ tự động đồng bộ kết quả về App Local.")